In [1]:
# import subtract_z_motion_neurons as szmn
import time
import numpy as np
import os
import glob
from pathlib import Path
from suite2p.registration import rigid
import numpy as np
from caiman.mmapping import load_memmap
import json
from tifffile import TiffFile, TiffWriter, imread
import bruker_concat_tif as ct
from scipy.ndimage import gaussian_filter, map_coordinates
import concurrent.futures
import compute_zcorr as cz

In [ ]:
import scipy.io

# Define the path to your .mat file
mat_file_path = "/home/wanglab/data/2P/C57_O1M2/10022023/run2/mesmerize/results_caiman.mat"

# Load the .mat file
mat_contents = scipy.io.loadmat(mat_file_path)

In [ ]:
for key in mat_contents:
    print(key)

In [ ]:
from joblib import Parallel, delayed

def fit_huber_regressor(i_pixel, Fz, Fz0, F, F0):
    huber = HuberRegressor(fit_intercept=False)
    try:
        huber.fit(Fz[i_pixel, :].reshape(-1, 1) - Fz0[i_pixel], F[i_pixel, :] - F0[i_pixel])
        b = huber.coef_[0]
    except ValueError as e:
        if "HuberRegressor convergence failed" in str(e):
            b = 0
            print(f"Convergence warning for pixel {i_pixel}. Setting b[i_pixel] to 0.")
        else:
            raise e
    return b

def subtract_z_motion_pixels(movie_mmap_path, FOV, zcorr, shifted_zstack_filename, zparams_path, n_jobs=-1):
    movie_16bit, dims, T = load_memmap(movie_mmap_path)
    pixels = np.reshape(movie_16bit.T, [T] + list(dims), order='F')
    Nneuron, Nx, Ny = pixels.shape

    if not isinstance(zparams_path, dict):
        with open(zparams_path, 'r') as file:
            params = json.load(file)

    F = pixels.reshape(Nneuron, Nx * Ny).T

    Nframe = Nneuron
    ROI_indices = np.arange(Nx * Ny)
    pix_y, pix_x = np.unravel_index(ROI_indices, (Nx, Ny))

    F0 = np.mean(F, axis=1)
    Fz0 = np.zeros(Nx * Ny)
    b = np.zeros(Nx * Ny)
    Fz = np.zeros((Nx * Ny, Nframe))
    Fz_rescaled = np.zeros((Nx * Ny, Nframe))
    Fcorrected = np.zeros((Nx * Ny, Nframe))

    info_zstack = Image.open(shifted_zstack_filename)
    Nz = info_zstack.n_frames
    Zstack = np.zeros((Ny, Nx, Nz), dtype=np.float32)
    info_zstack.close()

    zpos = np.argmax(convolve2d(zcorr, np.ones((1, 7)) / 7, mode='same'), axis=0)
    zpos_0 = mode(zpos).mode
    Zstack_0 = Zstack[:, :, zpos_0]

    shift, error, _ = phase_cross_correlation(Zstack_0, FOV, upsample_factor=10)

    tform = np.eye(3)
    tform[0, 2] = -shift[1]
    tform[1, 2] = -shift[0]

    for iz in range(Nz):
        Zstack[:, :, iz] = warp(Zstack[:, :, iz], tform)

    F810_t = Zstack[:, :, zpos]

    Fz = F810_t[pix_y, pix_x, :]
    Fz0 = np.mean(Fz, axis=1)

    b = Parallel(n_jobs=n_jobs)(delayed(fit_huber_regressor)(i_pixel, Fz, Fz0, F, F0) for i_pixel in tqdm(range(Nx * Ny), desc='Processing pixels'))
    b = np.array(b)

    Fz_rescaled = b[:, np.newaxis] * (Fz - Fz0[:, np.newaxis])
    Fcorrected = F - Fz_rescaled

    missing_F = np.abs(zpos - zpos_0) > 5
    Fcorrected[:, missing_F] = np.nan
    Fcorrected = pd.DataFrame(Fcorrected).interpolate(method='linear', axis=1, limit_direction='both').values

    Fz_scale_factor = np.mean(b)

    fig_z_drift = plt.figure()
    plt.plot(zpos)
    plt.xlabel('frames')
    plt.ylabel('z')

    Fcorrected_reshaped = Fcorrected.T.reshape(Nneuron, Nx, Ny)

    return Fcorrected_reshaped, b

In [ ]:
import subtract_z_motion_neurons as szmn
# F = temporal_components
# FOV=mean_map_motion_corrected
# for i_neuron = 1:Nneuron
#             ind = find(spatial_components(:,i_neuron)>0);
#             [ROI{i_neuron}.pix_x, ROI{i_neuron}.pix_y] = ind2sub([params.imaging.Npixel_x,params.imaging.Npixel_y],ind);
#             ROI{i_neuron}.pix_w = ones(length(ROI{i_neuron}.pix_x),1) / length(ROI{i_neuron}.pix_x);
#         end
# :+1:
# 1

# output: Fcorrected, b
# Fcorrected  size(Neurons,frames)
F=mat_contents['temporal_components']
FOV=mat_contents['mean_map_motion_corrected']
spatial_components=mat_contents['spatial_components']

class Components:
    def __init__(self, C, A):
        self.C = C
        self.A = A


components = Components(F, spatial_components)

zcorr=mat_contents['zcorr']
shifted_zstack_filename="/home/wanglab/data/2P/C57_O1M2/ZSeries-10022023-1300-004/zstack4_shifted.tif"
zparams_path="/home/wanglab/code/imaging_analysis/Analysis_2P/Mesmerize/parameters/params_zshift_default.json"

movie_mmap_path="/home/wanglab/data/2P/C57_O1M2/10022023/run2/mesmerize/edac2577-21c8-4a07-bae6-7c2be4732d05/edac2577-21c8-4a07-bae6-7c2be4732d05-cat_tiff_bt_els__d1_765_d2_765_d3_1_order_F_frames_1600.mmap"
Fcorrected_reshaped, b=cz.subtract_z_motion_pixels(movie_mmap_path, FOV, zcorr, shifted_zstack_filename,zparams_path)
